# Sheet 3.3: GRPO and GSPO for LLM reasoning

**Roadmap topic 3.3** (Fanout 07.05). Read first: DeepSeekMath section 4 (GRPO), DeepSeek-R1, GSPO, *Understanding R1-Zero-Like Training* (Dr. GRPO), DAPO, and the TRL `GRPOTrainer` docs (especially the "loss types" section).

You will write the core of a GRPO trainer: rewards, advantages, the loss, and the sequence-level GSPO ratio. Exercise 5 then trains a real model on a Colab T4.

**Rules:** fill every `# YOUR CODE` cell, never edit `CHECK` cells. Exercises 1-4 run on CPU.

In [ ]:
import math, re, torch, torch.nn.functional as F
torch.manual_seed(0)

## Exercise 1: A verifiable reward for GSM8K

Real model outputs are messy. Implement:
- `extract_answer(text)`: the final numeric answer. Priority: the content of the **last** `\boxed{...}`; else the number after the last `####`; else the last number in the text. Strip `$`, `,` thousands separators and trailing `.`; return a string like `"1234"` or `"-3.5"`, or `None`.
- `numbers_equal(a, b)`: numeric equality (so `"18"`, `"18.0"`, `"18.00"` match).
- `reward(completion, gold)`: 1.0 if correct, 0.0 otherwise. Add a **format** component: `+0.1` if the completion contains exactly one `\boxed{}`.

In [ ]:
def extract_answer(text):
    # YOUR CODE
    raise NotImplementedError
def numbers_equal(a, b):
    # YOUR CODE
    raise NotImplementedError
def reward(completion, gold):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    cases = {
        "So she has 3 + 15 = 18 apples. \\boxed{18}": "18",
        "The answer is $1,234.": "1234",
        "First 2 then 5. #### 7": "7",
        "Earlier \\boxed{3} was wrong; correct is \\boxed{-4.5}": "-4.5",
        "Total: \\boxed{\\$2,500}": "2500",
        "no numbers here": None,
        "It costs .5 dollars": ".5",
    }
    for t, exp in cases.items():
        got = extract_answer(t)
        assert (got is None and exp is None) or numbers_equal(got, exp), f"{t!r}: got {got}, want {exp}"
    assert numbers_equal("18", "18.00") and not numbers_equal("18", "81")
    gold = "Natalia sold 48/2 = 24 clips in May. 48+24 = 72. #### 72"
    assert reward("... so \\boxed{72}", gold) == 1.1 and reward("The answer is 72.", gold) == 1.0
    assert reward("\\boxed{71}", gold) == 0.1 and reward("\\boxed{72} or \\boxed{72}", gold) == 1.0
    print("PASS")
_check1()

**Written (1a):** Name two ways a policy could exploit this reward without solving the problem (think about what the extractor accepts), and how you'd harden it.

**Your answer:**

## Exercise 2: Group-relative advantages

Rewards come as a flat tensor of length `B·G`: `B` prompts, `G` completions each, grouped contiguously.
- `grpo_advantages(rewards, G, scale="std")`: `(r - mean_group) / (std_group + 1e-4)` with the **unbiased** std (as TRL uses `torch.std`), or with `scale="none"` just `r - mean_group` (Dr. GRPO).
- `frac_zero_std(rewards, G)`: fraction of groups where all rewards are equal. These groups produce zero advantage and no learning signal; DAPO's "dynamic sampling" filters them out.

In [ ]:
def grpo_advantages(rewards, G, scale="std"):
    # YOUR CODE
    raise NotImplementedError
def frac_zero_std(rewards, G):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    r = torch.tensor([1., 0., 0., 1., 1., 1., 1., 1., 0., 0., 0., 1.])
    a = grpo_advantages(r, 4)
    g0 = torch.tensor([1., 0., 0., 1.]); exp0 = (g0 - .5) / (g0.std() + 1e-4)
    assert torch.allclose(a[:4], exp0) and torch.allclose(a[4:8], torch.zeros(4)), "std-scaled advantages wrong"
    assert torch.allclose(grpo_advantages(r, 4, "none")[8:], torch.tensor([-.25, -.25, -.25, .75]))
    assert abs(frac_zero_std(r, 4) - 1 / 3) < 1e-6
    print("PASS")
_check2()

## Exercise 3: The GRPO loss, three ways

Inputs for a batch of `N = B·G` completions padded to length `T`:
`logp (N, T)` current policy token log-probs (requires grad), `old_logp`, `ref_logp`, `adv (N,)`, `mask (N, T)` (1 for completion tokens).

Per token: `ρ = exp(logp - old_logp)`, `L_tok = -min(ρ·A, clip(ρ, 1-ε, 1+ε)·A) + β·KL_k3`, with `KL_k3 = exp(ref - logp) - (ref - logp) - 1`.

Aggregate with `loss_type`:
- `"grpo"`: mean over each sequence's tokens, then mean over sequences (original paper; biased toward short wrong answers and long right ones)
- `"dapo"`: sum over all tokens / total number of tokens in the batch
- `"dr_grpo"`: sum over all tokens / (`N · max_len`), a constant normaliser

Return `(loss, stats)` where `stats` has `kl` (mean per token) and `clip_ratio`.

In [ ]:
def grpo_loss(logp, old_logp, ref_logp, adv, mask, eps=0.2, beta=0.04, loss_type="dapo", max_len=None):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    torch.manual_seed(0); N, T = 4, 6
    mask = torch.tensor([[1, 1, 1, 1, 1, 1], [1, 1, 0, 0, 0, 0], [1, 1, 1, 0, 0, 0], [1, 0, 0, 0, 0, 0]], dtype=torch.float32)
    old = torch.randn(N, T) - 2; adv = torch.tensor([1., -1., 0.5, 2.])
    for lt, expect in (("grpo", lambda: -adv[:, None] / mask.sum(1, keepdim=True) / N),
                       ("dapo", lambda: -adv[:, None].expand(N, T) / mask.sum()),
                       ("dr_grpo", lambda: -adv[:, None].expand(N, T) / (N * 8))):
        lp = old.clone().requires_grad_()
        loss, st = grpo_loss(lp, old, old, adv, mask, beta=0.04, loss_type=lt, max_len=8)
        loss.backward()
        assert torch.allclose(lp.grad, expect() * mask, atol=1e-6), f"{lt}: gradient at ratio=1, ref=policy is wrong"
        assert abs(st["kl"]) < 1e-7 and st["clip_ratio"] == 0
    lp = (old + torch.tensor([0.5, -0.5, 0.0, 0.0, 0.0, 0.0])).requires_grad_()
    loss, st = grpo_loss(lp, old, old, torch.ones(N), torch.ones(N, T), beta=0.0)
    loss.backward()
    assert torch.all(lp.grad[:, 0] == 0), "ratio above 1+eps with A>0 must be clipped (zero grad)"
    assert torch.all(lp.grad[:, 1] != 0), "ratio below 1-eps with A>0 is NOT clipped by min()"
    lp = old.clone().requires_grad_(); ref = old + 0.3
    _, st = grpo_loss(lp, old, ref, torch.zeros(N), mask)
    assert abs(st["kl"] - (math.exp(0.3) - 0.3 - 1)) < 1e-5
    print("PASS")
_check3()

**Written (3a):** Using the gradients the check verifies, explain the length bias of `"grpo"`: for a wrong answer (A < 0), does a long or a short completion get a smaller per-token penalty? What does that do to response length over training?

**Your answer:**

## Exercise 4: GSPO's sequence-level ratio

GSPO replaces per-token ratios with one ratio per sequence, the geometric mean: `s_i = exp( mean_t (logp - old_logp) )` over the sequence's real tokens, and clips `s_i` (with much smaller ε, e.g. 3e-4 / 4e-4 in the paper). Implement `gspo_ratio(logp, old_logp, mask)` and `gspo_loss(logp, old_logp, adv, mask, eps_low, eps_high)` (no KL term; mean over sequences of `-min(s·A, clip(s, 1-eps_low, 1+eps_high)·A)`).

In [ ]:
def gspo_ratio(logp, old_logp, mask):
    # YOUR CODE
    raise NotImplementedError
def gspo_loss(logp, old_logp, adv, mask, eps_low=3e-4, eps_high=4e-4):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 4 (do not edit) ----
def _check4():
    old = torch.zeros(2, 4); mask = torch.tensor([[1., 1., 1., 1.], [1., 1., 0., 0.]])
    lp = torch.tensor([[0.1, -0.1, 0.2, 0.0], [0.4, 0.0, 9.0, 9.0]])
    s = gspo_ratio(lp, old, mask)
    assert torch.allclose(s, torch.tensor([math.exp(0.05), math.exp(0.2)])), "padding must not count"
    lp = torch.zeros(2, 4, requires_grad=True)
    gspo_loss(lp, old, torch.tensor([1., -1.]), mask).backward()
    assert torch.allclose(lp.grad, torch.tensor([[-.125, -.125, -.125, -.125], [.25, .25, 0, 0]])), "gradient at s=1 wrong"
    print("PASS")
_check4()

**Written (4a):** Why does a per-token ratio become noisy for long sequences and MoE models (where routing can change between rollout and update), and how does the sequence-level ratio fix it?

---
## Exercise 5 (Colab T4): train a reasoner with GRPO

Model: `Qwen/Qwen2.5-0.5B-Instruct` (or `Qwen/Qwen3-0.6B` with thinking off). Data: `openai/gsm8k` (main). Use TRL's `GRPOTrainer` with LoRA (r = 16) and **your** `reward` function, `num_generations=8`, max completion length 256, `loss_type="dapo"` (also try `"grpo"`). Prompt the model to put its final answer in `\boxed{}`.

1. Before training: evaluate greedy accuracy on 500 GSM8K test questions with your `extract_answer`.
2. Train for as long as Colab allows (~300-600 steps). Log reward, completion length, KL, clip ratio, and `frac_zero_std`.
3. Evaluate again. **Target: an improvement whose paired-bootstrap 95% CI (sheet 0.3) excludes zero.**
4. Reward-hacking audit: read 50 high-reward completions and 20 where length grew most. Write what you find.
5. Unit-test your Exercise 3 loss against TRL: take one batch from the trainer, compute TRL's loss and yours with the same settings, and match them.

This run plus a short write-up is your **Step 3 portfolio artifact** (push the LoRA adapter and a model card to the Hugging Face Hub).

In [ ]:
# Colab: GRPO training with TRL

## Checkpoint
In writing: explain GRPO's length bias, how Dr. GRPO removes it, what DAPO's dynamic sampling and clip-higher do, and why GSPO moves the ratio to sequence level. One paragraph each, with the equation that changes.

---
### Hints for self-grading
- 1a: outputting many numbers so the "last number" fallback hits; putting several candidates in multiple `\boxed{}` if you took the first; boxing an expression the extractor reduces to the gold number. Harden by requiring exactly one box, checking the whole box content parses as a number, and rejecting very long outputs.
- 3a: with per-sequence averaging, each token of a long wrong answer is penalised less (1/len), so the model learns that wrong answers should be long; correct short answers get the largest per-token reward. Response length grows, especially for wrong answers.
- 4a: a per-token ratio is a product of many noisy per-token importance weights applied to a sequence-level reward; for long sequences the variance blows up, and routing changes in MoE make token likelihoods jumpy. The geometric mean over the sequence matches the unit of the reward and averages the noise out.